# Лабораторная 1: Transformer encoder для order-sensitive toy classification


## Что нужно знать до старта

Перед началом этой ЛР полезно открыть:
- [../README.md](./README.md)
- [guides/00_transformer_prerequisites.md](./guides/00_transformer_prerequisites.md)
- [guides/01_self_attention_and_positional_encoding_beginner.md](./guides/01_self_attention_and_positional_encoding_beginner.md)
- [guides/02_transformer_encoder_toy_walkthrough.md](./guides/02_transformer_encoder_toy_walkthrough.md)
- [../../02-Attention/theory/theory.md](../../02-Attention/theory/theory.md)

Это первая лабораторная блока `03-Transformer` и Шаг 5 общего курса.


## Выбор runtime

Здесь вы выбираете, где и на чём запускать notebook.

Что обычно выбирать:
- `auto` — лучший вариант по умолчанию. Если TensorFlow видит GPU, будет выбран GPU. Если GPU нет, notebook спокойно останется на CPU.
- `local-cpu` — локальный запуск только на CPU, даже если видеокарта есть.
- `local-gpu` — локальный запуск с обязательным GPU. Если GPU не настроен, notebook специально остановится с понятной ошибкой.
- `colab-cpu` / `colab-gpu` — запуск в Google Colab.
- `kaggle-cpu` / `kaggle-gpu` — запуск в Kaggle Notebooks.

Что важно:
- после изменения `RUNTIME_MODE` используйте `Restart & Run All`;
- `COURSE_REPO_HTTPS_URL` нужен только для Colab/Kaggle, если репозиторий ещё не клонирован в runtime;
- пока в ячейке стоит placeholder-URL, cloud auto-bootstrap не сможет сам скачать курс;
- guide `05` отвечает на вопрос, где и как запускать notebook;
- guide `06` нужен, если вы хотите именно локальный GPU и не уверены в версиях `TensorFlow` / `CUDA`;
- локальный GPU-path курса: `Linux + NVIDIA` или `Windows -> WSL2 + Ubuntu`;
- если `local-gpu` упирается в локальные CUDA/PTX ошибки, это обычно уже проблема GPU-стека, а не notebook. В таком случае спокойно переключайтесь на `local-cpu`, `colab-gpu` или `kaggle-gpu`.

Подробные guides:
- `themes/00-Foundations/guides/05_local_tensorflow_gpu_notebooks.md`
- `themes/00-Foundations/guides/06_tensorflow_cuda_version_selection.md`


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/03-Transformer/lab/requirements.txt"


def _detect_notebook_platform():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


## Интуиция задачи без формул

Здесь модель решает очень специальную задачу:
- во входной последовательности точно есть токены `7` и `3`;
- метка зависит не от того, что эти токены вообще были,
- а от того, **кто встретился раньше**.

Пример:

```text
[7, 5, 2, 3, PAD, PAD, ...] -> y = 1
[3, 5, 2, 7, PAD, PAD, ...] -> y = 0
```

Набор токенов одинаковый, но порядок разный.
Это и есть главная причина, по которой Transformer encoder не может обходиться без positional embedding.


## Как проходить эту ЛР без преподавателя

Фиксированный порядок для темы `03-Transformer`:
1. Прочитать `guides/00_transformer_prerequisites.md`.
2. Прочитать `guides/01_self_attention_and_positional_encoding_beginner.md`.
3. Пройти `guides/02_transformer_encoder_toy_walkthrough.md`.
4. Сделать свою первую попытку в этом starter notebook.
5. Если застряли, открыть `guides/04_transformer_debugging_playbook.md`.
6. Сделать вторую попытку после debugging-step.
7. Только потом смотреть в `solutions/01_transformer_encoder_order_toy_solution.ipynb`.


## Что изменилось после `02-Attention / ЛР01`

В attention-лаборатории decoder смотрел на encoder.

Теперь:
- рекуррентной ячейки больше нет;
- одна и та же последовательность сама строит `query`, `key`, `value`;
- вместо `cross-attention` мы используем `self-attention`;
- positional embedding становится обязательной частью модели.


## Контракт данных

Вход:
- padded последовательности целых token ids,
- форма `X -> (N, T)`.

Выход:
- бинарная метка `y -> (N,)`.

Правило метки:
- `y = 1`, если `7` стоит раньше `3`;
- `y = 0`, если `3` стоит раньше `7`.


## Таблица форм тензоров

| Сущность | Смысл | Форма |
|---|---|---|
| `X_train` | padded token ids | `(N, T)` |
| `padding_mask` | полезные позиции | `(N, T)` |
| `embeddings` | token + position embeddings | `(N, T, E)` |
| `attention_scores` | веса внимания по головам | `(N, H, T, T)` |
| `pooled` | один вектор на объект | `(N, E)` |
| `y_pred` | вероятность класса | `(N, 1)` |


## Шпаргалка по обозначениям и формам

- `N` — число объектов.
- `T` — длина последовательности после padding.
- `E` — размер embedding / model dimension.
- `H` — число attention heads.
- `PAD = 0`.

Практический фокус этой ЛР:
- проверить shapes;
- проверить mask;
- проверить, что attention не уходит в padded хвост.


## Контракт модели

Модель должна состоять из таких блоков:
1. `TokenAndPositionEmbedding`
2. `TransformerEncoderBlock`
3. masked average pooling
4. classifier head для binary classification

Отдельно нужен tracing-path, который возвращает `attention_scores` хотя бы для одного encoder block.


## Мини-теория

Минимальный encoder block:

$$
H_1 = \mathrm{LayerNorm}(X + \mathrm{MHA}(X))
$$

$$
H_2 = \mathrm{LayerNorm}(H_1 + \mathrm{FFN}(H_1))
$$

Если positional embedding отсутствует, то attention знает “какие токены были”, но слабо знает “где они были”.
Для этой лабораторной это критично, потому что метка зависит именно от порядка.


## Ручной разбор одного примера

Сравните две последовательности:

```text
A = [7, 5, 2, 3]
B = [3, 5, 2, 7]
```

Если смотреть только на набор токенов, они одинаковы.
Но по правилу задачи:
- `A -> 1`
- `B -> 0`

Значит, модели нужен позиционный сигнал.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
SEED = 7
PAD_ID = 0
KEY_TOKEN = 7
VALUE_TOKEN = 3
VOCAB_SIZE = 16
SEQ_LEN = 12
MIN_LEN = 4
TRAIN_SIZE = 4000
TEST_SIZE = 1000
EMBED_DIM = 32
NUM_HEADS = 2
FF_DIM = 64
BATCH_SIZE = 64
EPOCHS = 6

keras.utils.set_random_seed(SEED)
np.set_printoptions(linewidth=120)


## Контрольная точка 1: данные

На этом этапе нужно убедиться в трёх вещах:
- padded последовательности имеют общую длину `T`;
- токены `7` и `3` всегда присутствуют в полезной части;
- label зависит именно от порядка этих токенов.

Двигайтесь дальше только когда shape и rule label уже понятны руками.


In [ ]:
filler_tokens = np.array(
    [token for token in range(1, VOCAB_SIZE) if token not in (KEY_TOKEN, VALUE_TOKEN)],
    dtype=np.int32,
)


def generate_order_dataset(num_samples, seq_len=SEQ_LEN, min_len=MIN_LEN, seed=SEED):
    """Генерирует синтетический набор для задачи чувствительности к порядку.

    Аргументы:
      num_samples: Число последовательностей для генерации.
      seq_len: Максимальная длина последовательности после дополнения.
      min_len: Минимальная полезная длина до дополнения.
      seed: Зерно случайности для воспроизводимости.

    Возвращает:
      Кортеж `(X, y, lengths)`, где:
        `X`: Матрица токенов формы `(num_samples, seq_len)`.
        `y`: Вектор бинарных меток формы `(num_samples,)`.
        `lengths`: Полезные длины до дополнения формы `(num_samples,)`.

    Исключения:
      ValueError: Если `min_len` или `seq_len` заданы некорректно.
    """
    if min_len < 2 or seq_len < min_len:
        raise ValueError('Ожидается 2 <= min_len <= seq_len.')

    rng = np.random.default_rng(seed)
    X = np.full((num_samples, seq_len), PAD_ID, dtype=np.int32)
    y = np.zeros((num_samples,), dtype=np.int32)
    lengths = np.zeros((num_samples,), dtype=np.int32)

    for i in range(num_samples):
        # TODO 1.1: выберите полезную длину в диапазоне [min_len, seq_len]
        # Случайная длина от min_len до seq_len
        actual_len = rng.integers(min_len, seq_len + 1)
        lengths[i] = actual_len
        
        # TODO 1.2: соберите последовательность filler-токенов и вставьте KEY_TOKEN и VALUE_TOKEN
        # Создаем последовательность filler-токенов
        num_filler = actual_len - 2  # минус KEY_TOKEN и VALUE_TOKEN
        filler_seq = rng.choice(filler_tokens, size=num_filler, replace=True)
        
        # Выбираем две случайные позиции для KEY_TOKEN и VALUE_TOKEN
        positions = rng.choice(actual_len, size=2, replace=False)
        pos_key = min(positions)  # первая позиция (меньший индекс)
        pos_value = max(positions)  # вторая позиция (больший индекс)
        
        # Собираем последовательность
        seq = np.zeros(actual_len, dtype=np.int32)
        filler_idx = 0
        
        for pos in range(actual_len):
            if pos == pos_key:
                seq[pos] = KEY_TOKEN
            elif pos == pos_value:
                seq[pos] = VALUE_TOKEN
            else:
                seq[pos] = filler_seq[filler_idx]
                filler_idx += 1
        
        # TODO 1.3: посчитайте label по порядку этих токенов
        # Label = 1 если KEY_TOKEN встречается перед VALUE_TOKEN, иначе 0
        y[i] = 1 if pos_key < pos_value else 0
        
        # Записываем padded sequence в X
        X[i, :actual_len] = seq
    
    return X, y, lengths


X_all, y_all, lengths_all = generate_order_dataset(TRAIN_SIZE + TEST_SIZE)
X_train, X_test, y_train, y_test, len_train, len_test = train_test_split(
    X_all,
    y_all,
    lengths_all,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_all,
)

print('X_train shape:', X_train.shape)
print('X_test shape :', X_test.shape)
print('y_train mean :', y_train.mean())

# Дополнительная проверка корректности генерации
print("\nПроверка корректности сгенерированных данных:")
print(f"  Уникальные токены в X_train: {np.unique(X_train[X_train != PAD_ID])}")
print(f"  KEY_TOKEN={KEY_TOKEN}, VALUE_TOKEN={VALUE_TOKEN}")
print(f"  Доля label=1 в train: {y_train.mean():.3f}")
print(f"  Доля label=1 в test: {y_test.mean():.3f}")

# Проверка на примере
sample_idx = 0
sample_X = X_train[sample_idx]
sample_len = len_train[sample_idx]
sample_y = y_train[sample_idx]

print(f"\nПример последовательности {sample_idx}:")
print(f"  Длина: {sample_len}")
print(f"  Label: {sample_y} (KEY перед VALUE = {bool(sample_y)})")
print(f"  Последовательность: {sample_X[:sample_len]}")

# Находим позиции KEY и VALUE
key_positions = np.where(sample_X[:sample_len] == KEY_TOKEN)[0]
value_positions = np.where(sample_X[:sample_len] == VALUE_TOKEN)[0]

if len(key_positions) > 0 and len(value_positions) > 0:
    print(f"  KEY_TOKEN на позиции: {key_positions[0]}")
    print(f"  VALUE_TOKEN на позиции: {value_positions[0]}")
    print(f"  KEY перед VALUE: {key_positions[0] < value_positions[0]}")

# Проверка, что все последовательности содержат оба специальных токена
for i in range(min(5, len(X_train))):
    sample_len = len_train[i]
    has_key = KEY_TOKEN in X_train[i, :sample_len]
    has_value = VALUE_TOKEN in X_train[i, :sample_len]
    assert has_key and has_value, f"Пример {i} не содержит KEY или VALUE токенов!"
    
print("\n✓ Все проверки пройдены! Датасет сгенерирован корректно.")

In [ ]:
manual_a = np.array([[7, 5, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=np.int32)
manual_b = np.array([[3, 5, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=np.int32)

print('manual A label should be 1')
print('manual B label should be 0')
print('useful lengths sample:', len_train[:5])
print('class balance      :', np.bincount(y_train))


## Контрольная точка 2: embeddings + mask

Здесь нужно проверить две базовые идеи:
- `TokenAndPositionEmbedding` возвращает `(batch, time, embed_dim)`;
- padding mask сохраняется и потом попадёт в attention и pooling.

Если здесь shape или mask ведут себя странно, дальше в модели будет только больше шума.


In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    """Складывает токенные и позиционные векторы."""

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        if embed_dim < 1:
            raise ValueError('embed_dim должен быть положительным.')
        self.token_emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)
        self.supports_masking = True

    def call(self, inputs):
        """Возвращает сумму токенных и позиционных векторов."""
        # TODO 2.1: построить позиции через tf.range
        seq_len = tf.shape(inputs)[1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        
        # TODO 2.2: получить token embeddings и position embeddings
        token_embeddings = self.token_emb(inputs)
        position_embeddings = self.pos_emb(positions)
        
        # TODO 2.3: вернуть их сумму
        return token_embeddings + position_embeddings

    def compute_mask(self, inputs, mask=None):
        """Переадресует маску непустых токенов дальше по графу."""
        # TODO 2.4: вернуть mask от token embedding
        return self.token_emb.compute_mask(inputs, mask)


def masked_average(x, mask):
    """Вычисляет среднее по времени с учётом маски.

    Аргументы:
      x: Тензор признаков формы `(batch, time, embed_dim)`.
      mask: Булева маска полезных позиций формы `(batch, time)`.

    Возвращает:
      Усреднённый тензор формы `(batch, embed_dim)`.

    Исключения:
      ValueError: Если входной тензор имеет неверный ранг.
    """
    if x.shape.rank != 3:
        raise ValueError('Ожидается ранг 3 для тензора признаков.')
    mask = tf.cast(mask, x.dtype)
    mask = tf.expand_dims(mask, axis=-1)
    summed = tf.reduce_sum(x * mask, axis=1)
    counts = tf.reduce_sum(mask, axis=1)
    return summed / tf.maximum(counts, 1.0)


class TransformerEncoderBlock(layers.Layer):
    """Минимальный блок кодировщика трансформера."""

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)
        self.supports_masking = True

    def call(self, inputs, mask=None, training=None, return_attention_scores=False):
        """Выполняет прямой проход блока кодировщика.

        Аргументы:
          inputs: Входной тензор формы `(batch, time, embed_dim)`.
          mask: Булева маска формы `(batch, time)`.
          training: Признак режима обучения.
          return_attention_scores: Вернуть ли дополнительно веса внимания.

        Возвращает:
          Либо выходной тензор формы `(batch, time, embed_dim)`,
          либо кортеж `(output, attention_scores)`.

        Исключения:
          NotImplementedError: Пока шаг `TODO 3` не реализован.
        """
        # TODO 3.1: Соберите attention_mask как пересечение query и key масок.
        # Подсказка: query_mask = mask[:, :, None], key_mask = mask[:, None, :].
        attention_mask = None
        if mask is not None:
            # Создаем маску внимания: (batch, query_len, key_len)
            # True означает "можно обратить внимание"
            query_mask = tf.cast(mask[:, :, tf.newaxis], tf.bool)
            key_mask = tf.cast(mask[:, tf.newaxis, :], tf.bool)
            attention_mask = query_mask & key_mask
        
        # TODO 3.2: Вызовите self.att(..., return_attention_scores=...).
        attn_output, attn_scores = self.att(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
            return_attention_scores=True,
            training=training,
        )
        
        # TODO 3.3: Выполните residual + LayerNorm + FFN + residual + LayerNorm.
        # Первый residual блок (внимание)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # Второй residual блок (FFN)
        ffn_output = self.ffn(out1, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        
        if return_attention_scores:
            return out2, attn_scores
        return out2

    def compute_mask(self, inputs, mask=None):
        """Пробрасывает временную маску на следующий слой."""
        return mask


# Дополнительная проверка для убеждения, что все работает
def test_layers():
    """Тестирует реализованные слои на небольшом примере."""
    print("\nТестирование TokenAndPositionEmbedding...")
    vocab_size_test = 10
    embed_dim_test = 8
    maxlen_test = 16
    batch_size = 4
    
    embedding_layer = TokenAndPositionEmbedding(maxlen_test, vocab_size_test, embed_dim_test)
    test_input = tf.constant([[1, 2, 3, 0, 0], [4, 5, 0, 0, 0]], dtype=tf.int32)
    output = embedding_layer(test_input)
    
    print(f"  Входная форма: {test_input.shape}")
    print(f"  Выходная форма: {output.shape}")
    print(f"  Маска: {embedding_layer.compute_mask(test_input)}")
    assert output.shape == (2, 5, embed_dim_test), "Неверная форма выходного тензора!"
    print("  ✓ TokenAndPositionEmbedding работает корректно")
    
    print("\nТестирование masked_average...")
    x_test = tf.random.normal((batch_size, 10, embed_dim_test))
    mask_test = tf.cast(tf.random.uniform((batch_size, 10)) > 0.3, tf.bool)
    avg = masked_average(x_test, mask_test)
    print(f"  Входная форма x: {x_test.shape}")
    print(f"  Входная форма mask: {mask_test.shape}")
    print(f"  Выходная форма: {avg.shape}")
    assert avg.shape == (batch_size, embed_dim_test), "Неверная форма усреднения!"
    print("  ✓ masked_average работает корректно")
    
    print("\nТестирование TransformerEncoderBlock...")
    encoder_block = TransformerEncoderBlock(embed_dim_test, num_heads=2, ff_dim=16, rate=0.1)
    encoder_output = encoder_block(x_test, mask=mask_test, training=False)
    print(f"  Входная форма: {x_test.shape}")
    print(f"  Выходная форма: {encoder_output.shape}")
    assert encoder_output.shape == x_test.shape, "Неверная форма выходного тензора!"
    print("  ✓ TransformerEncoderBlock работает корректно")
    
    print("\n✅ Все тесты пройдены!")

# Запускаем тесты
test_layers()

In [ ]:
sample_tokens = X_train[:2]
sample_mask = sample_tokens != PAD_ID

sample_embedding_layer = TokenAndPositionEmbedding(SEQ_LEN, VOCAB_SIZE, EMBED_DIM)
sample_encoder_block = TransformerEncoderBlock(EMBED_DIM, NUM_HEADS, FF_DIM)

sample_embeddings = sample_embedding_layer(sample_tokens)
sample_encoded, sample_scores = sample_encoder_block(
    sample_embeddings,
    mask=sample_mask,
    return_attention_scores=True,
)

print('sample_embeddings:', sample_embeddings.shape)
print('sample_encoded   :', sample_encoded.shape)
print('sample_scores    :', sample_scores.shape)


## Контрольная точка 3: encoder block + classifier

После этого блока модель должна:
- принимать `tokens -> (batch, time)`;
- превращать их в embeddings с позициями;
- прогонять через `TransformerEncoderBlock`;
- сворачивать всё в одну вероятность `y_pred -> (batch, 1)` через `sigmoid`.


In [ ]:
keras.utils.set_random_seed(SEED)

transformer_inputs = keras.Input(shape=(SEQ_LEN,), dtype='int32', name='tokens')
padding_mask = layers.Lambda(lambda x: tf.not_equal(x, PAD_ID), name='padding_mask')(transformer_inputs)

# TODO 4.1: создать TokenAndPositionEmbedding и TransformerEncoderBlock
# Параметры модели
VOCAB_SIZE = len(filler_tokens) + 3  # filler_tokens + KEY_TOKEN + VALUE_TOKEN + PAD_ID
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_ENCODER_LAYERS = 2
DROPOUT_RATE = 0.1

# Создаем слои
embedding_layer = TokenAndPositionEmbedding(
    maxlen=SEQ_LEN,
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    name='token_position_embedding'
)

# Создаем несколько encoder блоков
encoder_blocks = []
for i in range(NUM_ENCODER_LAYERS):
    encoder_blocks.append(
        TransformerEncoderBlock(
            embed_dim=EMBED_DIM,
            num_heads=NUM_HEADS,
            ff_dim=FF_DIM,
            rate=DROPOUT_RATE,
            name=f'transformer_encoder_{i}'
        )
    )

# TODO 4.2: прогнать inputs через embedding и encoder block
x = embedding_layer(transformer_inputs)

# Прогоняем через все encoder блоки
for encoder_block in encoder_blocks:
    x = encoder_block(x, mask=padding_mask, training=True)

# TODO 4.3: сделать masked average pooling, небольшой Dense classifier head и финальный sigmoid
# Masked average pooling
x = masked_average(x, padding_mask)

# Dense classifier head
x = layers.Dense(32, activation='relu', name='dense_1')(x)
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(16, activation='relu', name='dense_2')(x)
x = layers.Dropout(DROPOUT_RATE)(x)

# Финальный sigmoid для бинарной классификации
outputs = layers.Dense(1, activation='sigmoid', name='classifier_output')(x)

# Создаем модель
model = keras.Model(inputs=transformer_inputs, outputs=outputs, name='transformer_classifier')

# TODO 4.4: скомпилировать model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall(), 
             keras.metrics.AUC(name='auc')]
)

# Выводим архитектуру модели
print("Архитектура модели:")
model.summary()

# Визуализация модели (если нужно)
try:
    keras.utils.plot_model(model, 'transformer_classifier.png', show_shapes=True, show_dtype=True)
    print("\nВизуализация модели сохранена в 'transformer_classifier.png'")
except Exception as e:
    print(f"\nВизуализация модели не удалась: {e}")

# Проверяем, что модель работает на небольшом батче
print("\nПроверка модели на тестовом батче...")
test_batch = X_train[:4]
predictions = model.predict(test_batch, verbose=0)
print(f"  Входная форма: {test_batch.shape}")
print(f"  Выходная форма: {predictions.shape}")
print(f"  Предсказания (первые 4): {predictions.flatten()}")
print(f"  Истинные метки: {y_train[:4]}")

# Дополнительная проверка маскировки
print("\nПроверка работы padding mask...")
sample_with_pad = np.array([[1, 2, 3, 0, 0, 0, 0, 0, 0, 0]])  # SEQ_LEN=10 для примера
mask_output = padding_mask(sample_with_pad)
print(f"  Вход: {sample_with_pad[0]}")
print(f"  Маска: {mask_output.numpy()[0]}")
print(f"  PAD_ID={PAD_ID}, маска игнорирует нули: {not mask_output.numpy()[0, 3]}")

print("\n✅ Модель успешно собрана и скомпилирована!")

In [ ]:
model.summary()


## Контрольная точка 4: обучение

Перед запуском `fit` проверьте:
- classifier head действительно бинарный;
- loss = `binary_crossentropy`;
- train и validation будут сравниваться отдельно от финального test.


In [ ]:
# TODO 5.1: обучить model.fit(...) с validation_split
# TODO 5.2: сохранить результат в history

# Параметры обучения
BATCH_SIZE = 64
EPOCHS = 50
VALIDATION_SPLIT = 0.2  # 20% данных для валидации
LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 5
REDUCE_LR_PATIENCE = 3

# Создаем колбэки для улучшения обучения
callbacks = [
    # Ранняя остановка, чтобы избежать переобучения
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1
    ),
    # Уменьшение learning rate при застревании
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=REDUCE_LR_PATIENCE,
        min_lr=1e-6,
        verbose=1
    ),
    # Сохранение лучшей модели
    keras.callbacks.ModelCheckpoint(
        'best_transformer_classifier.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Логирование
    keras.callbacks.CSVLogger('training_log.csv')
]

print("="*70)
print("ОБУЧЕНИЕ МОДЕЛИ ТРАНСФОРМЕРА ДЛЯ ЗАДАЧИ ЧУВСТВИТЕЛЬНОСТИ К ПОРЯДКУ")
print("="*70)
print(f"Размер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Validation split: {VALIDATION_SPLIT}")
print(f"Learning rate: {LEARNING_RATE}")
print("="*70)

# Обучаем модель
history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "="*70)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО")
print("="*70)

# Выводим лучшие результаты
best_epoch = np.argmin(history.history['val_loss']) + 1
best_val_loss = min(history.history['val_loss'])
best_val_accuracy = max(history.history['val_accuracy'])
best_val_auc = max(history.history.get('val_auc', [0]))

print(f"\nЛучшие результаты на валидации:")
print(f"  Эпоха: {best_epoch}")
print(f"  Loss: {best_val_loss:.4f}")
print(f"  Accuracy: {best_val_accuracy:.4f}")
print(f"  AUC: {best_val_auc:.4f}")

# Визуализация процесса обучения
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# График потерь
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_title('Model Loss', fontsize=14)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# График точности
axes[0, 1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 1].set_title('Model Accuracy', fontsize=14)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# График precision
if 'precision' in history.history:
    axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
    axes[1, 0].plot(history.history['val_precision'], label='Validation Precision', linewidth=2)
    axes[1, 0].set_title('Precision', fontsize=14)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

# График recall
if 'recall' in history.history:
    axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
    axes[1, 1].plot(history.history['val_recall'], label='Validation Recall', linewidth=2)
    axes[1, 1].set_title('Recall', fontsize=14)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Оценка на тестовой выборке
print("\n" + "="*70)
print("ОЦЕНКА НА ТЕСТОВОЙ ВЫБОРКЕ")
print("="*70)

test_results = model.evaluate(X_test, y_test, verbose=0)
metrics_names = model.metrics_names

print("\nРезультаты:")
for name, value in zip(metrics_names, test_results):
    print(f"  {name}: {value:.4f}")

test_loss = test_results[metrics_names.index('loss')]
test_accuracy = test_results[metrics_names.index('accuracy')]

# Дополнительные метрики для бинарной классификации
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

y_pred_prob = model.predict(X_test, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Визуализация confusion matrix
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Class 0', 'Class 1'])
plt.yticks(tick_marks, ['Class 0', 'Class 1'])
plt.xlabel('Predicted')
plt.ylabel('True')

# Добавляем значения в ячейки
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

# ROC кривая
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nTest AUC: {roc_auc:.4f}")

# Проверка, что модель обучилась лучше случайного угадывания
random_baseline = 0.5
if test_accuracy > random_baseline:
    print(f"\n✅ Модель обучилась успешно! Accuracy ({test_accuracy:.4f}) > случайного угадывания ({random_baseline})")
else:
    print(f"\n⚠️ Модель не превзошла случайное угадывание. Accuracy: {test_accuracy:.4f}")

# Сохраняем модель
model.save('transformer_classifier_final.keras')
print("\n✅ Модель сохранена как 'transformer_classifier_final.keras'")

# Анализ важности позиций (примерно)
print("\n" + "="*70)
print("АНАЛИЗ РАБОТЫ МОДЕЛИ")
print("="*70)

# Тестируем на нескольких примерах
print("\nПримеры предсказаний на тестовых данных:")
for i in range(min(10, len(X_test))):
    pred_prob = model.predict(X_test[i:i+1], verbose=0)[0, 0]
    pred_class = 1 if pred_prob > 0.5 else 0
    true_class = y_test[i]
    
    # Находим позиции KEY и VALUE
    seq = X_test[i]
    key_pos = np.where(seq == KEY_TOKEN)[0]
    value_pos = np.where(seq == VALUE_TOKEN)[0]
    
    if len(key_pos) > 0 and len(value_pos) > 0:
        order = "KEY before VALUE" if key_pos[0] < value_pos[0] else "VALUE before KEY"
        correct = "✓" if pred_class == true_class else "✗"
        print(f"  {correct} Пример {i+1}: {order} -> pred={pred_class} (prob={pred_prob:.3f}), true={true_class}")
    else:
        print(f"  Пример {i+1}: pred={pred_class} (prob={pred_prob:.3f}), true={true_class}")

print("\n" + "="*70)
print("ОБУЧЕНИЕ ЗАВЕРШЕНО УСПЕШНО!")
print("="*70)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()


## Контрольная точка 5: attention trace и критерии завершения

Перед сдачей здесь должны одновременно выполняться все условия:
- `test_acc >= 0.95`;
- два ручных примера с перестановкой `7` и `3` дают разные предсказания;
- heatmap строится только по non-PAD части последовательности.

Сначала снимите итоговую метрику и ручные примеры, потом интерпретируйте attention trace.


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'test_loss = {test_loss:.4f}')
print(f'test_acc  = {test_acc:.4f}')


In [ ]:
paired_examples = np.array(
    [
        [7, 5, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0],
        [3, 5, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    dtype=np.int32,
)

paired_probs = model.predict(paired_examples, verbose=0).ravel()
for seq, prob in zip(paired_examples, paired_probs):
    label = int(prob >= 0.5)
    print(seq, '-> prob=', round(float(prob), 4), 'label=', label)


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'test_loss = {test_loss:.4f}')
print(f'test_acc  = {test_acc:.4f}')

paired_examples = np.array(
    [
        [7, 5, 2, 3, 0, 0, 0, 0, 0, 0, 0, 0],
        [3, 5, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0],
    ],
    dtype=np.int32,
)

paired_probs = model.predict(paired_examples, verbose=0).ravel()
for seq, prob in zip(paired_examples, paired_probs):
    label = int(prob >= 0.5)
    print(seq, '-> prob=', round(float(prob), 4), 'label=', label)

# TODO 6.1: взять один тестовый пример
print("\n" + "="*70)
print("ТРАССИРОВКА ATTENTION SCORES")
print("="*70)

# Выбираем один тестовый пример
sample_idx = 0
sample_input = X_test[sample_idx:sample_idx+1]  # Форма (1, SEQ_LEN)
sample_label = y_test[sample_idx]
sample_length = len_test[sample_idx] if 'len_test' in dir() else SEQ_LEN

print(f"Тестовый пример {sample_idx}:")
print(f"  Последовательность: {sample_input[0][:sample_length]}")
print(f"  Истинная метка: {sample_label}")
print(f"  Длина (без padding): {sample_length}")

# Находим позиции KEY и VALUE токенов
key_positions = np.where(sample_input[0] == KEY_TOKEN)[0]
value_positions = np.where(sample_input[0] == VALUE_TOKEN)[0]

if len(key_positions) > 0 and len(value_positions) > 0:
    print(f"  KEY_TOKEN ({KEY_TOKEN}) на позиции: {key_positions[0]}")
    print(f"  VALUE_TOKEN ({VALUE_TOKEN}) на позиции: {value_positions[0]}")
    print(f"  Порядок: {'KEY before VALUE' if key_positions[0] < value_positions[0] else 'VALUE before KEY'}")

# TODO 6.2: получить attention_scores через tracing-path/model

# Функция для создания модели, возвращающей attention scores из encoder блоков
def build_attention_tracing_model(base_model, encoder_block_index=-1):
    """Создает модель, которая возвращает attention_scores из указанного encoder блока."""
    # Находим все TransformerEncoderBlock слои
    encoder_blocks = [layer for layer in base_model.layers if isinstance(layer, TransformerEncoderBlock)]
    
    if not encoder_blocks:
        raise ValueError("В модели нет TransformerEncoderBlock слоев")
    
    # Берем последний блок по умолчанию
    target_block = encoder_blocks[encoder_block_index]
    
    # Строим граф для получения attention scores
    input_layer = base_model.input
    x = input_layer
    
    # Проходим через все слои до целевого блока
    attention_scores = None
    for layer in base_model.layers:
        if layer == target_block:
            # Для целевого блока вызываем с return_attention_scores=True
            x, attention_scores = layer(x, return_attention_scores=True)
        elif isinstance(layer, TransformerEncoderBlock):
            # Для остальных encoder блоков без attention scores
            x = layer(x)
        elif layer.name != base_model.layers[0].name:  # Пропускаем input слой
            x = layer(x)
    
    if attention_scores is None:
        raise ValueError("Не удалось получить attention scores")
    
    tracing_model = keras.Model(inputs=input_layer, outputs=attention_scores, name='attention_tracer')
    return tracing_model

# Строим trace-модель для последнего encoder блока
try:
    attention_tracer = build_attention_tracing_model(model, encoder_block_index=-1)
    print("\n✓ Trace-модель для диагностики внимания создана")
    print(f"  Выходная форма: attention_scores (batch, num_heads, query_len, key_len)")
except Exception as e:
    print(f"✗ Ошибка при создании trace-модели: {e}")
    # Альтернативный подход: если модель уже имеет слои с attention
    # Пытаемся получить attention scores через внутренние слои
    attention_tracer = None
    print("  Пробуем альтернативный способ...")

# Получаем attention scores
if attention_tracer is not None:
    attention_scores = attention_tracer(sample_input, training=False)
    print(f"\nAttention scores shape: {attention_scores.shape}")
else:
    # Альтернативный способ: получить attention scores из промежуточных слоев
    # Создаем модель, которая возвращает выходы всех encoder блоков
    intermediate_model = keras.Model(
        inputs=model.input,
        outputs=[layer.output for layer in model.layers if isinstance(layer, TransformerEncoderBlock)]
    )
    encoder_outputs = intermediate_model(sample_input, training=False)
    # Для простоты, создаем заглушку
    print("  Внимание: используется упрощенная диагностика")
    attention_scores = None

# TODO 6.3: усреднить attention по головам и обрезать padded хвост

if attention_scores is not None:
    # Усредняем по головам
    avg_attention = tf.reduce_mean(attention_scores, axis=1)  # (batch, query_len, key_len)
    avg_attention = avg_attention[0].numpy()  # Убираем batch dimension
    
    # Обрезаем до реальной длины (без padding)
    trimmed_attention = avg_attention[:sample_length, :sample_length]
    
    print(f"\nОбрезанная attention карта формы: {trimmed_attention.shape}")
    
    # Визуализация attention карты
    plt.figure(figsize=(10, 8))
    plt.imshow(trimmed_attention, cmap='Blues', aspect='auto')
    plt.colorbar(label='Attention weight')
    plt.xlabel('Key positions (context)')
    plt.ylabel('Query positions (current)')
    plt.title(f'Attention Map (Encoder Block)\nSequence length: {sample_length}')
    
    # Отмечаем позиции KEY и VALUE токенов
    if len(key_positions) > 0 and len(value_positions) > 0:
        plt.axhline(y=key_positions[0], color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'KEY at {key_positions[0]}')
        plt.axhline(y=value_positions[0], color='green', linestyle='--', linewidth=1, alpha=0.7, label=f'VALUE at {value_positions[0]}')
        plt.axvline(x=key_positions[0], color='red', linestyle='--', linewidth=1, alpha=0.7)
        plt.axvline(x=value_positions[0], color='green', linestyle='--', linewidth=1, alpha=0.7)
        plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Анализ внимания к KEY и VALUE токенам
    print("\nАнализ внимания к специальным токенам:")
    
    # Внимание к KEY_TOKEN
    if len(key_positions) > 0:
        key_attn = trimmed_attention[:, key_positions[0]]
        print(f"  Среднее внимание к KEY_TOKEN (поз.{key_positions[0]}): {key_attn.mean():.4f}")
        print(f"  Максимальное внимание к KEY_TOKEN: {key_attn.max():.4f}")
    
    # Внимание к VALUE_TOKEN
    if len(value_positions) > 0:
        value_attn = trimmed_attention[:, value_positions[0]]
        print(f"  Среднее внимание к VALUE_TOKEN (поз.{value_positions[0]}): {value_attn.mean():.4f}")
        print(f"  Максимальное внимание к VALUE_TOKEN: {value_attn.max():.4f}")
    
    # Визуализация внимания к определенным позициям
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    if len(key_positions) > 0:
        axes[0].plot(range(sample_length), trimmed_attention[:, key_positions[0]], 'b-o', linewidth=2, markersize=4)
        axes[0].set_title(f'Attention to KEY_TOKEN at position {key_positions[0]}')
        axes[0].set_xlabel('Query position')
        axes[0].set_ylabel('Attention weight')
        axes[0].grid(True, alpha=0.3)
    
    if len(value_positions) > 0:
        axes[1].plot(range(sample_length), trimmed_attention[:, value_positions[0]], 'g-o', linewidth=2, markersize=4)
        axes[1].set_title(f'Attention to VALUE_TOKEN at position {value_positions[0]}')
        axes[1].set_xlabel('Query position')
        axes[1].set_ylabel('Attention weight')
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Сравнение внимания для пары примеров (KEY before VALUE vs VALUE before KEY)
    print("\n" + "="*70)
    print("СРАВНЕНИЕ ATTENTION ДЛЯ ПАРНЫХ ПРИМЕРОВ")
    print("="*70)
    
    # Используем paired_examples для сравнения
    paired_attentions = []
    for seq in paired_examples:
        seq_input = seq.reshape(1, -1)
        if attention_tracer is not None:
            attn_scores = attention_tracer(seq_input, training=False)
            avg_attn = tf.reduce_mean(attn_scores, axis=1)[0].numpy()
            
            # Находим реальную длину (без нулей)
            real_len = np.sum(seq != 0)
            trimmed = avg_attn[:real_len, :real_len]
            paired_attentions.append(trimmed)
            
            print(f"\nПоследовательность: {seq[:real_len]}")
            print(f"  Форма attention: {trimmed.shape}")
            print(f"  Средний вес внимания: {trimmed.mean():.4f}")
            print(f"  Максимальный вес: {trimmed.max():.4f}")
    
    # Сравнение распределения внимания
    if len(paired_attentions) == 2:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        for idx, (attn, seq) in enumerate(zip(paired_attentions, paired_examples)):
            real_len = np.sum(seq != 0)
            axes[idx].imshow(attn, cmap='Blues', aspect='auto')
            axes[idx].set_title(f'Sequence {idx+1}: {seq[:real_len]}')
            axes[idx].set_xlabel('Key positions')
            axes[idx].set_ylabel('Query positions')
        
        plt.tight_layout()
        plt.show()

else:
    print("\n⚠️ Не удалось получить attention scores для детального анализа")
    print("  Модель может не поддерживать return_attention_scores или слои переопределены")

# Альтернативный анализ: влияние позиций на предсказание
print("\n" + "="*70)
print("АНАЛИЗ ВЛИЯНИЯ ПОЗИЦИЙ ТОКЕНОВ")
print("="*70)

# Создаем тестовые примеры с разными позициями KEY и VALUE
test_sequences = []
positions_to_test = [(1, 2), (1, 3), (2, 3), (2, 4), (3, 5)]

for key_pos, value_pos in positions_to_test:
    seq = np.zeros(SEQ_LEN, dtype=np.int32)
    seq[key_pos] = KEY_TOKEN
    seq[value_pos] = VALUE_TOKEN
    # Заполняем filler токенами остальные позиции (кроме первой)
    filler_idx = 0
    for i in range(SEQ_LEN):
        if i != key_pos and i != value_pos and i > 0:
            seq[i] = filler_tokens[filler_idx % len(filler_tokens)]
            filler_idx += 1
    test_sequences.append(seq)

test_sequences = np.array(test_sequences)
predictions = model.predict(test_sequences, verbose=0)

print("\nВлияние порядка и расстояния между KEY и VALUE:")
for (key_pos, value_pos), pred in zip(positions_to_test, predictions):
    order = "KEY→VALUE" if key_pos < value_pos else "VALUE→KEY"
    distance = abs(value_pos - key_pos)
    prob = pred[0]
    label = "Class 1" if prob >= 0.5 else "Class 0"
    print(f"  KEY at {key_pos}, VALUE at {value_pos}: {order}, dist={distance} -> prob={prob:.4f} ({label})")

print("\n✅ Диагностика attention завершена!")

In [ ]:
token_labels = [str(token) for token in sample_tokens[0][:sample_length]]

plt.figure(figsize=(6, 5))
plt.imshow(mean_attention, cmap='magma', aspect='auto')
plt.colorbar(label='attention weight')
plt.xticks(range(sample_length), token_labels)
plt.yticks(range(sample_length), token_labels)
plt.xlabel('key positions')
plt.ylabel('query positions')
plt.title('Mean self-attention over heads')
plt.tight_layout()
plt.show()


## Опционально после сдачи: почему здесь не хватает bag-of-words логики

## Если не получилось с первого раза

Не начинайте с гипотезы “Transformer слишком сложный”.
Сначала проверьте:
- rule label;
- shapes;
- mask;
- позиционное кодирование;
- форму `attention_scores`.


## Если застряли: порядок диагностики

1. Проверить два ручных примера с одинаковым набором токенов.
2. Проверить форму `embeddings`.
3. Проверить форму `attention_scores`.
4. Проверить, что attention не смотрит на `PAD`.
5. Только потом смотреть в solution notebook.


## Чек-лист перед сдачей

- Данные padded до общей длины `T`.
- `7` и `3` всегда есть в полезной части последовательности.
- `TokenAndPositionEmbedding` возвращает `(batch, time, embed_dim)`.
- `TransformerEncoderBlock` использует mask.
- Модель возвращает бинарную вероятность `y_pred -> (N, 1)` через `sigmoid`.
- `test_acc >= 0.95`.
- Два ручных примера с перестановкой `7` и `3` различаются по предсказанию.
- Attention trace визуализируется хотя бы на одном примере и обрезан до non-PAD части.


## Как использовать решение без самообмана

Правильный порядок:
1. первая самостоятельная попытка;
2. walkthrough;
3. debugging playbook;
4. вторая самостоятельная попытка;
5. только потом solution notebook.

Если просто скопировать custom layer, то можно получить зелёные ячейки без понимания, зачем нужны positional embedding и padding mask.


## Опционально после сдачи: мини-экзамен

1. Почему self-attention без позиции не решает эту задачу надёжно?
2. Зачем здесь нужна padding mask, если это не `seq2seq`?
3. Почему `attention_scores` имеют две оси времени?
4. Что именно делает residual connection в encoder block?


## Опционально после сдачи: что дальше

Эта лабораторная закрывает synthetic-вход в `Transformer encoder`.

Следующий шаг курса — `03-Transformer / ЛР02`, где тот же блок переносится на реальный `IMDB`:
- вход уже не synthetic sequence, а review;
- метка остаётся одна на последовательность;
- self-attention и positional embedding остаются центральными.

Открывать дальше: [02_transformer_encoder_imdb.ipynb](./02_transformer_encoder_imdb.ipynb)


## Опционально после сдачи: вопросы для самопроверки

## Типичные ошибки (симптом -> причина -> исправление)

- `attention_scores` неожиданной формы -> перепутано ожидание выхода `MultiHeadAttention` -> помнить про `(batch, heads, T, T)`.
- Accuracy около `0.5` -> метка считается не по позиции, а по наличию токенов -> проверить rule label.
- Heatmap яркий на padded хвосте -> mask не передан в attention -> проверить `padding_mask`.
- Порядок не влияет на модель -> забыли positional embedding -> проверить `TokenAndPositionEmbedding`.
